In [2]:
# imports and config
from pathlib import Path
import math
import heapq
import re
import warnings 
import numpy as np
import pandas as pd
import searoute as sr
import country_converter as coco
import folium
import geopandas as gpd 
from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
# paths and base dimensions
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

FACT_ROOT = PROJECT_ROOT / "data" / "silver" / "comtrade" / "comtrade_fact"
FACT_FILES = sorted(FACT_ROOT.rglob("ref_year=*/reporter_iso3=*/*.parquet"))

DIMENSIONS_ROOT = PROJECT_ROOT / "data" / "silver" / "comtrade" / "dimensions"
DIMENSIONS_ROOT.mkdir(parents=True, exist_ok=True)

DIM_COUNTRY_PATH = DIMENSIONS_ROOT / "dim_country.parquet"
DIM_OUTPUT_PATH = PROJECT_ROOT / "data" / "silver" / "comtrade" / "dim_trade_routes.parquet"
COUNTRY_PORT_DIM_PATH = DIMENSIONS_ROOT / "dim_country_ports.parquet"
PORT_BASIN_DIM_PATH = DIMENSIONS_ROOT / "dim_port_basin.parquet"
ROUTE_APPLICABILITY_PATH = DIMENSIONS_ROOT / "bridge_country_route_applicability.parquet"
CHOKEPOINT_DIM_PATH = DIMENSIONS_ROOT / "dim_chokepoint.parquet"
BASIN_EDGE_PATH = DIMENSIONS_ROOT / "bridge_basin_graph_edges.parquet"
BASIN_PATH_RULES_PATH = DIMENSIONS_ROOT / "bridge_port_basin_chokepoints.parquet"
TRANSHIPMENT_HUB_PATH = DIMENSIONS_ROOT / "dim_transshipment_hub.parquet"
BASIN_HUB_BRIDGE_PATH = DIMENSIONS_ROOT / "bridge_basin_default_hubs.parquet"

PORT_INDEX_PATH = PROJECT_ROOT / "ingest" / "Kaggle" / "world_port_index" / "UpdatedPub150.csv"

FULL_REBUILD_DIM_TRADE_ROUTES = True
ROUTE_SCENARIO = "default_shortest"
# supported examples:
# - default_shortest
# - suez_disrupted
# - panama_disrupted
# - cape_preferred
# - risk_avoidance

MAX_PORTS_DEFAULT = 3
MAX_PORTS_MULTI_BASIN = 8

dim_country = pd.read_parquet(DIM_COUNTRY_PATH)

if DIM_OUTPUT_PATH.exists():
    dim_trade_routes_existing = pd.read_parquet(DIM_OUTPUT_PATH)
else:
    dim_trade_routes_existing = pd.DataFrame()

In [8]:
# build a route applicability bridge from fact motCode values
if not FACT_FILES:
    raise FileNotFoundError(f"No parquet fact files found under: {FACT_ROOT}")

silver_df = pd.concat([pd.read_parquet(path) for path in FACT_FILES], ignore_index=True)

silver_grain = [
    "period",
    "reporter_iso3",
    "partner_iso3",
    "flowCode",
    "cmdCode",
    "classification_version",
    "customsCode",
    "motCode",
    "partner2Code",
]

silver_df.drop_duplicates(subset=silver_grain, inplace=True)

code_to_iso3 = {}
if "country_code" in dim_country.columns and "iso3" in dim_country.columns:
    code_to_iso3 = dict(
        zip(
            pd.to_numeric(dim_country["country_code"], errors="coerce").astype("Int64"),
            dim_country["iso3"].astype("string").str.upper().str.strip(),
        )
    )

def _normalize_iso3(series: pd.Series) -> pd.Series:
    clean = series.astype("string").str.upper().str.strip()
    clean = clean.replace({"": pd.NA, "NAN": pd.NA, "NONE": pd.NA, "W00": pd.NA})
    valid = clean.str.fullmatch(r"[A-Z]{3}", na=False)
    return clean.where(valid, pd.NA)

silver_df["reporter_iso3"] = _normalize_iso3(silver_df["reporter_iso3"])
silver_df["partner_iso3"] = _normalize_iso3(silver_df["partner_iso3"])
silver_df["partner2_iso3"] = pd.to_numeric(silver_df["partner2Code"], errors="coerce").astype("Int64").map(code_to_iso3)
silver_df["partner2_iso3"] = _normalize_iso3(silver_df["partner2_iso3"])
silver_df["motCode"] = pd.to_numeric(silver_df["motCode"], errors="coerce").astype("Int64")

SEA_CODES = {2100}
INLAND_WATER_CODES = {2200}
UNKNOWN_CODES = {0, 9000, 9900}
NON_MARINE_CODES = {1000, 3100, 3200, 9100, 9110, 9200, 9300}

route_candidates = (
    silver_df
    .dropna(subset=["reporter_iso3", "partner_iso3"])
    .groupby(["reporter_iso3", "partner_iso3", "partner2_iso3"], dropna=False)
    .agg(
        row_count=("partner_iso3", "size"),
        trade_value_usd=("trade_value_usd", "sum"),
        has_sea=("motCode", lambda s: s.dropna().isin(SEA_CODES).any()),
        has_inland_water=("motCode", lambda s: s.dropna().isin(INLAND_WATER_CODES).any()),
        has_unknown=("motCode", lambda s: s.dropna().isin(UNKNOWN_CODES).any()),
        has_non_marine=("motCode", lambda s: s.dropna().isin(NON_MARINE_CODES).any()),
        mot_codes_seen=("motCode", lambda s: "|".join(sorted({str(int(x)) for x in s.dropna().unique()}))),
    )
    .reset_index()
)

route_candidates["route_applicability_status"] = np.select(
    [
        route_candidates["has_sea"],
        (~route_candidates["has_sea"]) & route_candidates["has_inland_water"],
        (~route_candidates["has_sea"]) & (~route_candidates["has_inland_water"]) & route_candidates["has_unknown"],
        (~route_candidates["has_sea"]) & (~route_candidates["has_inland_water"]) & route_candidates["has_non_marine"],
    ],
    [
        "MARITIME_ELIGIBLE",
        "INLAND_WATER_ONLY",
        "UNKNOWN_ONLY",
        "NON_MARITIME_ONLY",
    ],
    default="NO_MODE_EVIDENCE"
)

route_candidates.to_parquet(ROUTE_APPLICABILITY_PATH, index=False)

display(route_candidates["route_applicability_status"].value_counts(dropna=False).rename_axis("route_applicability_status").reset_index(name="pair_count"))
display(route_candidates.head(10))

,route_applicability_status,pair_count
0,UNKNOWN_ONLY,3274
1,MARITIME_ELIGIBLE,1688
2,INLAND_WATER_ONLY,25


,reporter_iso3,partner_iso3,partner2_iso3,row_count,trade_value_usd,has_sea,has_inland_water,has_unknown,has_non_marine,mot_codes_seen,route_applicability_status
0,BGR,AGO,<NA>,16,1.117763e+06,True,False,True,True,0|1000|2100,MARITIME_ELIGIBLE
1,BGR,ALB,<NA>,528,6.187100e+08,True,False,True,True,0|1000|2100|3200,MARITIME_ELIGIBLE
2,BGR,ARE,<NA>,250,1.508292e+08,True,True,True,True,0|1000|2100|2200|3200,MARITIME_ELIGIBLE
3,BGR,ARG,<NA>,341,1.413164e+07,True,False,True,True,0|1000|2100|3100|3200,MARITIME_ELIGIBLE
4,BGR,ARM,<NA>,238,1.966293e+07,True,False,True,True,0|2100|3200,MARITIME_ELIGIBLE
5,BGR,ATG,<NA>,24,5.069904e+05,True,False,True,True,0|2100|3200,MARITIME_ELIGIBLE
6,BGR,AUS,<NA>,106,3.912566e+06,True,False,True,True,0|1000|2100|3200,MARITIME_ELIGIBLE
7,BGR,AUT,<NA>,1890,3.797258e+08,False,True,True,True,0|1000|2200|3100|3200|9200,INLAND_WATER_ONLY
8,BGR,AZE,<NA>,250,6.370388e+08,True,False,True,True,0|1000|2100|3100|3200,MARITIME_ELIGIBLE
9,BGR,BEL,<NA>,1394,6.942962e+08,True,False,True,True,0|1000|2100|3100|3200|9200,MARITIME_ELIGIBLE


In [13]:
# build dim_country_ports and dim_port_basin from World Port Index using richer basin inference
port_raw = pd.read_csv(PORT_INDEX_PATH, low_memory=False)

port_candidates = port_raw.rename(
    columns={
        "Main Port Name": "main_port_name",
        "Alternate Port Name": "alternate_port_name",
        "Country Code": "country_label",
        "World Water Body": "world_water_body",
        "Harbor Size": "harbor_size",
        "Harbor Type": "harbor_type",
        "Harbor Use": "harbor_use",
        "Facilities - Container": "fac_container",
        "Facilities - Solid Bulk": "fac_solid_bulk",
        "Facilities - Liquid Bulk": "fac_liquid_bulk",
        "Facilities - Oil Terminal": "fac_oil_terminal",
        "Facilities - LNG Terminal": "fac_lng_terminal",
        "Latitude": "latitude",
        "Longitude": "longitude",
    }
).copy()

port_candidates["port_name"] = (
    port_candidates["main_port_name"].astype("string").str.strip()
    .fillna(port_candidates["alternate_port_name"].astype("string").str.strip())
)

port_candidates["country_label"] = port_candidates["country_label"].astype("string").str.strip()
port_candidates["world_water_body"] = port_candidates["world_water_body"].astype("string").str.strip()
port_candidates["harbor_size"] = port_candidates["harbor_size"].astype("string").str.strip()
port_candidates["harbor_type"] = port_candidates["harbor_type"].astype("string").str.strip()
port_candidates["harbor_use"] = port_candidates["harbor_use"].astype("string").str.strip()

port_candidates["latitude"] = pd.to_numeric(port_candidates["latitude"], errors="coerce")
port_candidates["longitude"] = pd.to_numeric(port_candidates["longitude"], errors="coerce")

port_candidates["iso3"] = coco.convert(
    names=port_candidates["country_label"].fillna(""),
    to="ISO3",
    not_found="not found",
)
port_candidates["iso3"] = pd.Series(port_candidates["iso3"], index=port_candidates.index).replace("not found", pd.NA)

def _as_txt(value) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip().lower()

def infer_port_basin(world_water_body: str, latitude: float = np.nan, longitude: float = np.nan) -> str:
    txt = _as_txt(world_water_body)

    if any(k in txt for k in ["black sea", "sea of azov"]): return "BLACK_SEA"
    if any(k in txt for k in ["mediterranean", "aegean", "adriatic", "ionian", "tyrrhenian", "ligurian"]): return "MEDITERRANEAN"
    if any(k in txt for k in ["north sea", "english channel", "bay of biscay"]): return "NORTH_ATLANTIC_EUROPE"
    if "baltic" in txt: return "BALTIC"
    if any(k in txt for k in ["persian gulf", "arabian gulf"]): return "GULF"
    if "red sea" in txt: return "RED_SEA"
    if "arabian sea" in txt: return "ARABIAN_SEA"
    if "indian ocean" in txt: return "INDIAN_OCEAN"
    if any(k in txt for k in ["south china sea", "east china sea", "yellow sea", "sea of japan"]): return "WESTERN_PACIFIC"
    if any(k in txt for k in ["north pacific ocean", "bering sea", "sea of okhotsk", "tatar strait"]): return "PACIFIC"
    if "gulf of mexico" in txt: return "NORTH_AMERICA_ATLANTIC"
    if any(k in txt for k in ["caribbean", "west indies"]): return "CARIBBEAN"
    if "south atlantic ocean" in txt: return "SOUTH_ATLANTIC"
    if "north atlantic ocean" in txt:
        if pd.notna(longitude) and longitude < -30: return "NORTH_AMERICA_ATLANTIC"
        return "ATLANTIC"
    if "atlantic ocean" in txt:
        if pd.notna(latitude) and latitude < 0: return "SOUTH_ATLANTIC"
        if pd.notna(longitude) and longitude < -30: return "NORTH_AMERICA_ATLANTIC"
        return "ATLANTIC"
    if any(k in txt for k in ["gulf of guinea", "bight of benin", "bight of bonny"]): return "WEST_AFRICA_ATLANTIC"
    if any(k in txt for k in ["mozambique channel", "somali basin"]): return "EAST_AFRICA_INDIAN"
    if any(k in txt for k in ["south pacific ocean", "humboldt current"]): return "SOUTH_AMERICA_PACIFIC"
    if any(k in txt for k in ["great lakes", "danube", "river", "lake "]): return "INLAND_WATERWAY"
    return "UNKNOWN_COASTAL"

port_candidates["port_basin"] = port_candidates.apply(
    lambda row: infer_port_basin(row["world_water_body"], row["latitude"], row["longitude"]),
    axis=1
)

size_rank = {"Very Small": 1, "Small": 2, "Medium": 3, "Large": 4}
yes_rank = {"Yes": 1, "No": 0, "Unknown": 0}

for col in ["fac_container", "fac_solid_bulk", "fac_liquid_bulk", "fac_oil_terminal", "fac_lng_terminal"]:
    port_candidates[col] = port_candidates[col].map(yes_rank).fillna(0).astype(int)

port_candidates["size_rank"] = port_candidates["harbor_size"].map(size_rank).fillna(0).astype(int)

port_candidates["port_score"] = (
    port_candidates["size_rank"] * 10
    + port_candidates["fac_container"] * 5
    + port_candidates["fac_solid_bulk"] * 4
    + port_candidates["fac_liquid_bulk"] * 4
    + port_candidates["fac_oil_terminal"] * 6
    + port_candidates["fac_lng_terminal"] * 6
)

usable_ports = (
    port_candidates
    .dropna(subset=["iso3", "port_name", "latitude", "longitude"])
    .drop_duplicates(subset=["iso3", "port_name", "latitude", "longitude"])
    .copy()
)

# ---------------------------------------------------------
# DYNAMIC MULTI-BASIN & MEGA-PORT SELECTION 
# ---------------------------------------------------------

# 1. Get Top 2 ports per Basin (Guarantees coastal diversity for RUS, USA, CAN, etc.)
ranked_by_basin = usable_ports.sort_values(
    ["iso3", "port_basin", "port_score", "size_rank", "port_name"],
    ascending=[True, True, False, False, True]
).copy()
ranked_by_basin["basin_rank"] = ranked_by_basin.groupby(["iso3", "port_basin"]).cumcount() + 1
diverse_ports = ranked_by_basin[ranked_by_basin["basin_rank"] <= 2].copy()

# 2. Get Top 5 ports Overall per Country (Guarantees mega-exporters like CHN, NLD get enough capacity)
ranked_overall = usable_ports.sort_values(
    ["iso3", "port_score", "size_rank", "port_name"],
    ascending=[True, False, False, True]
).copy()
ranked_overall["overall_rank"] = ranked_overall.groupby("iso3").cumcount() + 1
mega_ports = ranked_overall[ranked_overall["overall_rank"] <= 5].copy()

# 3. Combine sets and deduplicate 
combined_ports = pd.concat([diverse_ports, mega_ports]).drop_duplicates(subset=["iso3", "port_name"])

# 4. Re-apply a clean port_rank for the final dimension table
combined_ports = combined_ports.sort_values(
    ["iso3", "port_score", "size_rank", "port_name"],
    ascending=[True, False, False, True]
).copy()
combined_ports["port_rank"] = combined_ports.groupby("iso3").cumcount() + 1
# ---------------------------------------------------------

dim_country_ports = combined_ports[
    [
        "iso3", "port_name", "latitude", "longitude", "world_water_body", "port_basin",
        "harbor_size", "harbor_type", "harbor_use", "fac_container", "fac_solid_bulk",
        "fac_liquid_bulk", "fac_oil_terminal", "fac_lng_terminal", "port_score", "port_rank"
    ]
].copy()

dim_port_basin = (
    dim_country_ports[["port_basin", "world_water_body"]]
    .drop_duplicates()
    .sort_values(["port_basin", "world_water_body"])
    .copy()
)

dim_country_ports.to_parquet(COUNTRY_PORT_DIM_PATH, index=False)
dim_port_basin.to_parquet(PORT_BASIN_DIM_PATH, index=False)

display(dim_country_ports.head(20))
display(dim_country_ports.loc[dim_country_ports["iso3"].isin(["USA", "CHN", "RUS", "NLD"])].sort_values(["iso3", "port_rank"]))

Wake Island not found in regex
Johnson Atoll not found in regex
Midway Islands not found in regex
 not found in regex
 not found in regex
 not found in regex
 not found in regex


,iso3,port_name,latitude,longitude,world_water_body,port_basin,harbor_size,harbor_type,harbor_use,fac_container,fac_solid_bulk,fac_liquid_bulk,fac_oil_terminal,fac_lng_terminal,port_score,port_rank
563,ABW,Sint Nicolaas Baai,12.433333,-69.916667,Caribbean Sea; North Atlantic Ocean,CARIBBEAN,Small,Coastal (Breakwater),Unknown,0,0,0,0,0,20,1
3146,ABW,Paarden Baai - (Oranjestad),12.516667,-70.033333,Caribbean Sea; North Atlantic Ocean,CARIBBEAN,Very Small,Coastal (Breakwater),Unknown,0,0,0,0,0,10,2
512,AGO,Cabinda,-5.533333,12.200000,South Atlantic Ocean,SOUTH_ATLANTIC,Small,Open Roadstead,Unknown,0,0,0,0,0,20,1
1783,AGO,Estrela Oil Field,-6.450000,12.400000,South Atlantic Ocean,SOUTH_ATLANTIC,Small,Open Roadstead,Unknown,0,0,0,0,0,20,2
281,AGO,Lobito,-12.333333,13.566667,South Atlantic Ocean,SOUTH_ATLANTIC,Small,Coastal (Natural),Unknown,0,0,0,0,0,20,3
2523,AGO,Luanda,-8.800000,13.250000,South Atlantic Ocean,SOUTH_ATLANTIC,Small,Coastal (Natural),Unknown,0,0,0,0,0,20,4
1331,AGO,Malongo Oil Terminal,-5.433333,12.083333,South Atlantic Ocean,SOUTH_ATLANTIC,Small,Open Roadstead,Unknown,0,0,0,0,0,20,5
1416,ALB,Durres,41.316667,19.450000,Adriatic Sea; Mediterranean Sea; North Atlanti...,MEDITERRANEAN,Small,Coastal (Breakwater),Unknown,0,0,0,0,0,20,1
2726,ALB,Shengjin,41.816667,19.600000,Adriatic Sea; Mediterranean Sea; North Atlanti...,MEDITERRANEAN,Very Small,Coastal (Natural),Unknown,0,0,0,0,0,10,2
2638,ALB,Vlores,40.466667,19.500000,Adriatic Sea; Mediterranean Sea; North Atlanti...,MEDITERRANEAN,Very Small,Coastal (Natural),Unknown,0,0,0,0,0,10,3


,iso3,port_name,latitude,longitude,world_water_body,port_basin,harbor_size,harbor_type,harbor_use,fac_container,fac_solid_bulk,fac_liquid_bulk,fac_oil_terminal,fac_lng_terminal,port_score,port_rank
271,CHN,Dalian,38.916667,121.666667,Yellow Sea; North Pacific Ocean,WESTERN_PACIFIC,Large,Coastal (Breakwater),Unknown,0,0,0,0,0,40,1
1807,CHN,Lon Shui Terminal,20.833333,115.683333,South China Sea; North Pacific Ocean,WESTERN_PACIFIC,Large,Open Roadstead,Unknown,0,0,0,0,0,40,2
674,CHN,Qingdao Gang,36.033333,120.266667,Yellow Sea; North Pacific Ocean,WESTERN_PACIFIC,Large,Open Roadstead,Unknown,0,0,0,0,0,40,3
2829,CHN,Shanghai,31.216667,121.500000,East China Sea; North Pacific Ocean,WESTERN_PACIFIC,Large,River (Natural),Unknown,0,0,0,0,0,40,4
2846,CHN,Tianjin Xin Gang,38.966667,117.833333,Bo Hai; Yellow Sea; North Pacific Ocean,WESTERN_PACIFIC,Large,River (Natural),Unknown,0,0,0,0,0,40,5
2506,CHN,Xiamen,24.450000,118.066667,Taiwan Strait; North Pacific Ocean,PACIFIC,Medium,River (Natural),Unknown,0,0,0,0,0,30,6
679,CHN,Quanzhou,24.883333,118.600000,Taiwan Strait; North Pacific Ocean,PACIFIC,Small,Coastal (Natural),Unknown,0,0,0,0,0,20,7
1451,NLD,Amsterdam,52.366667,4.900000,North Sea; North Atlantic Ocean,NORTH_ATLANTIC_EUROPE,Large,Canal or Lake,Unknown,0,0,0,0,0,40,1
2599,NLD,Rotterdam,51.900000,4.483333,North Sea; North Atlantic Ocean,NORTH_ATLANTIC_EUROPE,Large,River (Basins),Unknown,0,0,0,0,0,40,2
914,NLD,Dordrecht,51.816667,4.650000,North Sea; North Atlantic Ocean,NORTH_ATLANTIC_EUROPE,Medium,River (Natural),Unknown,0,0,0,0,0,30,3


In [15]:
# coverage check for ports, landlocked handling, and dynamic gateway logic
country_port_coverage = (
    dim_country_ports.groupby("iso3", as_index=False)["port_rank"]
    .count()
    .rename(columns={"port_rank": "ports_found"})
)

# Added 'is_eu' to leverage your dimensional tagging
country_geo = dim_country[["iso3", "country_name", "continent", "region", "subregion", "is_eu"]].drop_duplicates().copy()
country_geo = country_geo.merge(country_port_coverage[["iso3", "ports_found"]], on="iso3", how="left")

# Safely handle numeric and boolean NAs to avoid evaluation errors
country_geo["ports_found"] = country_geo["ports_found"].fillna(0).astype(int)
country_geo["has_wpi_ports"] = country_geo["ports_found"] > 0
country_geo["is_eu"] = country_geo["is_eu"].fillna(False).astype(bool)

print("Dynamically computing landlocked gateways via spatial topologies...")

# Load standard low-resolution world map directly from Natural Earth's cloud storage
url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(url)

# Bulletproof column extraction (handles Natural Earth schema drift)
iso_col = 'ADM0_A3' if 'ADM0_A3' in world.columns else 'ISO_A3' if 'ISO_A3' in world.columns else 'ISO_A3_EH'
name_col = 'NAME' if 'NAME' in world.columns else 'ADMIN'

world['iso_a3'] = world[iso_col].astype(str)
world['name'] = world[name_col].astype(str)

# Fix ALL known missing ISO3 codes in Natural Earth (-99 values)
world.loc[world['name'] == 'France', 'iso_a3'] = 'FRA'
world.loc[world['name'] == 'Norway', 'iso_a3'] = 'NOR'
world.loc[world['name'] == 'Somaliland', 'iso_a3'] = 'SOM'
world.loc[world['name'] == 'Kosovo', 'iso_a3'] = 'RKS'
world.loc[world['name'] == 'N. Cyprus', 'iso_a3'] = 'CYP' 

# Merge our port flags into the spatial dataframe
world = world.merge(
    country_geo[['iso3', 'has_wpi_ports']], 
    left_on='iso_a3', right_on='iso3', how='left'
)
# Explicit boolean cast to prevent DataFrame indexing errors
world['has_wpi_ports'] = world['has_wpi_ports'].fillna(False).astype(bool)

LANDLOCKED_GATEWAYS = {}
landlocked_iso3s = country_geo[~country_geo['has_wpi_ports']]['iso3'].tolist()

for iso in landlocked_iso3s:
    # Attempt to find the geometry for the current country
    country_geom_series = world[world['iso_a3'] == iso]['geometry']
    
    # If it's a non-spatial group code (e.g., EUR, F19, S19), geometry will be empty
    if country_geom_series.empty:
        group_info = country_geo[country_geo['iso3'] == iso]
        if not group_info.empty:
            info = group_info.iloc[0]
            
            # Dynamic aggregation based on your dimension tags
            if info['is_eu']:
                gateways = country_geo[(country_geo['is_eu']) & (country_geo['has_wpi_ports'])]['iso3'].tolist()
            elif pd.notna(info['region']) and info['region'] not in ['World', 'Special']:
                gateways = country_geo[(country_geo['region'] == info['region']) & (country_geo['has_wpi_ports'])]['iso3'].tolist()
            elif info['region'] == 'World':
                gateways = ['CHN', 'USA', 'NLD', 'SGP', 'ARE', 'ZAF', 'BRA']
            else:
                gateways = []
                
            if gateways:
                LANDLOCKED_GATEWAYS[iso] = gateways
        continue
        
    geom = country_geom_series.iloc[0]
    
    # 1-Hop: Find direct neighbors (countries whose borders intersect)
    neighbors = world[world.geometry.intersects(geom) & (world['iso_a3'] != iso)]
    coastal_neighbors = neighbors[neighbors['has_wpi_ports']]['iso_a3'].dropna().tolist()
    
    if coastal_neighbors:
        LANDLOCKED_GATEWAYS[iso] = coastal_neighbors
    else:
        # 2-Hop: For doubly-landlocked countries
        if not neighbors.empty:
            hop2_neighbors = world[world.geometry.intersects(neighbors.geometry.unary_union) & (world['iso_a3'] != iso)]
            hop2_coastal = hop2_neighbors[hop2_neighbors['has_wpi_ports']]['iso_a3'].dropna().tolist()
            if hop2_coastal:
                LANDLOCKED_GATEWAYS[iso] = hop2_coastal

# Flag the countries that were dynamically assigned a gateway
country_geo["is_landlocked_assumed"] = country_geo["iso3"].isin(LANDLOCKED_GATEWAYS.keys())
COUNTRY_FLAGS = country_geo.set_index("iso3")[["country_name", "is_landlocked_assumed", "has_wpi_ports", "continent", "region", "subregion"]].to_dict("index")

print(f"Successfully computed coastal gateways for {len(LANDLOCKED_GATEWAYS)} entities.")

display(
    country_geo[
        ["iso3", "country_name", "ports_found", "has_wpi_ports", "is_landlocked_assumed"]
    ].sort_values(["is_landlocked_assumed", "ports_found", "iso3"], ascending=[False, True, True]).head(40)
)

Dynamically computing landlocked gateways via spatial topologies...
Successfully computed coastal gateways for 56 entities.


/var/folders/rg/5ylghp0s7jn56p9q3z8vtvt00000gn/T/ipykernel_19133/3843512145.py:83: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  hop2_neighbors = world[world.geometry.intersects(neighbors.geometry.unary_union) & (world['iso_a3'] != iso)]


,iso3,country_name,ports_found,has_wpi_ports,is_landlocked_assumed
0,A79,"LAIA, nes",0,False,True
2,AFG,Afghanistan,0,False,True
4,AIA,Anguilla,0,False,True
6,AND,Andorra,0,False,True
9,ARM,Armenia,0,False,True
15,AUT,Austria,0,False,True
16,AZE,Azerbaijan,0,False,True
17,BDI,Burundi,0,False,True
21,BFA,Burkina Faso,0,False,True
27,BLM,Saint Barthélemy,0,False,True


In [16]:
# persist chokepoint master data and create a scenario-aware basin graph
dim_chokepoint = pd.DataFrame(
    [
        {"chokepoint_name": "Turkish Straits", "longitude": 29.0, "latitude": 41.0, "kind": "strait"},
        {"chokepoint_name": "Suez Canal", "longitude": 32.5498, "latitude": 30.8167, "kind": "canal"},
        {"chokepoint_name": "Hormuz Strait", "longitude": 56.25, "latitude": 26.57, "kind": "strait"},
        {"chokepoint_name": "Bab el-Mandeb", "longitude": 43.33, "latitude": 12.58, "kind": "strait"},
        {"chokepoint_name": "Panama Canal", "longitude": -79.58, "latitude": 9.08, "kind": "canal"},
        {"chokepoint_name": "Malacca Strait", "longitude": 99.8, "latitude": 2.5, "kind": "strait"},
        {"chokepoint_name": "Gibraltar Strait", "longitude": -5.6, "latitude": 35.95, "kind": "strait"},
        {"chokepoint_name": "Cape of Good Hope", "longitude": 18.47, "latitude": -34.36, "kind": "cape"},
        {"chokepoint_name": "Open Sea", "longitude": np.nan, "latitude": np.nan, "kind": "open_sea"},
    ]
)

dim_chokepoint.to_parquet(CHOKEPOINT_DIM_PATH, index=False)

# weighted graph edges between basins
basin_graph_edges = pd.DataFrame(
    [
        # enclosed / european
        {"origin_basin": "BLACK_SEA", "destination_basin": "MEDITERRANEAN", "chokepoint_name": "Turkish Straits", "base_cost": 2.0},
        {"origin_basin": "MEDITERRANEAN", "destination_basin": "BLACK_SEA", "chokepoint_name": "Turkish Straits", "base_cost": 2.0},
        {"origin_basin": "MEDITERRANEAN", "destination_basin": "NORTH_ATLANTIC_EUROPE", "chokepoint_name": "Gibraltar Strait", "base_cost": 2.0},
        {"origin_basin": "NORTH_ATLANTIC_EUROPE", "destination_basin": "MEDITERRANEAN", "chokepoint_name": "Gibraltar Strait", "base_cost": 2.0},
        {"origin_basin": "MEDITERRANEAN", "destination_basin": "ATLANTIC", "chokepoint_name": "Gibraltar Strait", "base_cost": 2.0},
        {"origin_basin": "ATLANTIC", "destination_basin": "MEDITERRANEAN", "chokepoint_name": "Gibraltar Strait", "base_cost": 2.0},

        # suez corridor
        {"origin_basin": "MEDITERRANEAN", "destination_basin": "RED_SEA", "chokepoint_name": "Suez Canal", "base_cost": 2.0},
        {"origin_basin": "RED_SEA", "destination_basin": "MEDITERRANEAN", "chokepoint_name": "Suez Canal", "base_cost": 2.0},
        {"origin_basin": "RED_SEA", "destination_basin": "INDIAN_OCEAN", "chokepoint_name": "Bab el-Mandeb", "base_cost": 2.0},
        {"origin_basin": "INDIAN_OCEAN", "destination_basin": "RED_SEA", "chokepoint_name": "Bab el-Mandeb", "base_cost": 2.0},
        {"origin_basin": "ARABIAN_SEA", "destination_basin": "GULF", "chokepoint_name": "Hormuz Strait", "base_cost": 2.0},
        {"origin_basin": "GULF", "destination_basin": "ARABIAN_SEA", "chokepoint_name": "Hormuz Strait", "base_cost": 2.0},
        {"origin_basin": "INDIAN_OCEAN", "destination_basin": "ARABIAN_SEA", "chokepoint_name": "Open Sea", "base_cost": 1.0},
        {"origin_basin": "ARABIAN_SEA", "destination_basin": "INDIAN_OCEAN", "chokepoint_name": "Open Sea", "base_cost": 1.0},

        # asia corridor
        {"origin_basin": "INDIAN_OCEAN", "destination_basin": "WESTERN_PACIFIC", "chokepoint_name": "Malacca Strait", "base_cost": 2.5},
        {"origin_basin": "WESTERN_PACIFIC", "destination_basin": "INDIAN_OCEAN", "chokepoint_name": "Malacca Strait", "base_cost": 2.5},

        # atlantic / pacific
        {"origin_basin": "ATLANTIC", "destination_basin": "PACIFIC", "chokepoint_name": "Panama Canal", "base_cost": 3.0},
        {"origin_basin": "PACIFIC", "destination_basin": "ATLANTIC", "chokepoint_name": "Panama Canal", "base_cost": 3.0},
        {"origin_basin": "NORTH_AMERICA_ATLANTIC", "destination_basin": "PACIFIC", "chokepoint_name": "Panama Canal", "base_cost": 3.0},
        {"origin_basin": "PACIFIC", "destination_basin": "NORTH_AMERICA_ATLANTIC", "chokepoint_name": "Panama Canal", "base_cost": 3.0},
        {"origin_basin": "NORTH_AMERICA_ATLANTIC", "destination_basin": "ATLANTIC", "chokepoint_name": "Open Sea", "base_cost": 1.0},
        {"origin_basin": "ATLANTIC", "destination_basin": "NORTH_AMERICA_ATLANTIC", "chokepoint_name": "Open Sea", "base_cost": 1.0},
        {"origin_basin": "CARIBBEAN", "destination_basin": "ATLANTIC", "chokepoint_name": "Open Sea", "base_cost": 1.0},
        {"origin_basin": "ATLANTIC", "destination_basin": "CARIBBEAN", "chokepoint_name": "Open Sea", "base_cost": 1.0},

        # cape alternative / south atlantic
        {"origin_basin": "INDIAN_OCEAN", "destination_basin": "SOUTH_ATLANTIC", "chokepoint_name": "Cape of Good Hope", "base_cost": 4.5},
        {"origin_basin": "SOUTH_ATLANTIC", "destination_basin": "INDIAN_OCEAN", "chokepoint_name": "Cape of Good Hope", "base_cost": 4.5},
        {"origin_basin": "SOUTH_ATLANTIC", "destination_basin": "ATLANTIC", "chokepoint_name": "Open Sea", "base_cost": 1.0},
        {"origin_basin": "ATLANTIC", "destination_basin": "SOUTH_ATLANTIC", "chokepoint_name": "Open Sea", "base_cost": 1.0},
        {"origin_basin": "WEST_AFRICA_ATLANTIC", "destination_basin": "ATLANTIC", "chokepoint_name": "Open Sea", "base_cost": 1.0},
        {"origin_basin": "ATLANTIC", "destination_basin": "WEST_AFRICA_ATLANTIC", "chokepoint_name": "Open Sea", "base_cost": 1.0},
        {"origin_basin": "EAST_AFRICA_INDIAN", "destination_basin": "INDIAN_OCEAN", "chokepoint_name": "Open Sea", "base_cost": 1.0},
        {"origin_basin": "INDIAN_OCEAN", "destination_basin": "EAST_AFRICA_INDIAN", "chokepoint_name": "Open Sea", "base_cost": 1.0},

        # same-ocean connective tissue
        {"origin_basin": "WESTERN_PACIFIC", "destination_basin": "PACIFIC", "chokepoint_name": "Open Sea", "base_cost": 1.0},
        {"origin_basin": "PACIFIC", "destination_basin": "WESTERN_PACIFIC", "chokepoint_name": "Open Sea", "base_cost": 1.0},
        {"origin_basin": "BALTIC", "destination_basin": "NORTH_ATLANTIC_EUROPE", "chokepoint_name": "Open Sea", "base_cost": 1.0},
        {"origin_basin": "NORTH_ATLANTIC_EUROPE", "destination_basin": "BALTIC", "chokepoint_name": "Open Sea", "base_cost": 1.0},
    ]
)

def apply_scenario_weights(edges: pd.DataFrame, route_scenario: str) -> pd.DataFrame:
    edges = edges.copy()
    edges["scenario_cost"] = edges["base_cost"]

    if route_scenario == "suez_disrupted":
        edges.loc[edges["chokepoint_name"].isin(["Suez Canal", "Bab el-Mandeb"]), "scenario_cost"] += 8.0
    elif route_scenario == "panama_disrupted":
        edges.loc[edges["chokepoint_name"].eq("Panama Canal"), "scenario_cost"] += 8.0
    elif route_scenario == "cape_preferred":
        edges.loc[edges["chokepoint_name"].isin(["Suez Canal", "Bab el-Mandeb"]), "scenario_cost"] += 3.0
        edges.loc[edges["chokepoint_name"].eq("Cape of Good Hope"), "scenario_cost"] -= 1.5
    elif route_scenario == "risk_avoidance":
        edges.loc[edges["chokepoint_name"].isin(["Hormuz Strait", "Bab el-Mandeb", "Suez Canal"]), "scenario_cost"] += 4.0

    return edges

basin_graph_edges = apply_scenario_weights(basin_graph_edges, ROUTE_SCENARIO)
basin_graph_edges.to_parquet(BASIN_EDGE_PATH, index=False)

display(basin_graph_edges.head(20))

,origin_basin,destination_basin,chokepoint_name,base_cost,scenario_cost
0,BLACK_SEA,MEDITERRANEAN,Turkish Straits,2.0,2.0
1,MEDITERRANEAN,BLACK_SEA,Turkish Straits,2.0,2.0
2,MEDITERRANEAN,NORTH_ATLANTIC_EUROPE,Gibraltar Strait,2.0,2.0
3,NORTH_ATLANTIC_EUROPE,MEDITERRANEAN,Gibraltar Strait,2.0,2.0
4,MEDITERRANEAN,ATLANTIC,Gibraltar Strait,2.0,2.0
5,ATLANTIC,MEDITERRANEAN,Gibraltar Strait,2.0,2.0
6,MEDITERRANEAN,RED_SEA,Suez Canal,2.0,2.0
7,RED_SEA,MEDITERRANEAN,Suez Canal,2.0,2.0
8,RED_SEA,INDIAN_OCEAN,Bab el-Mandeb,2.0,2.0
9,INDIAN_OCEAN,RED_SEA,Bab el-Mandeb,2.0,2.0


In [17]:
# optional transshipment hub support
dim_transshipment_hub = pd.DataFrame(
    [
        {"hub_port": "Rotterdam", "hub_iso3": "NLD", "hub_basin": "NORTH_ATLANTIC_EUROPE", "hub_rank": 1},
        {"hub_port": "Piraeus", "hub_iso3": "GRC", "hub_basin": "MEDITERRANEAN", "hub_rank": 1},
        {"hub_port": "Algeciras", "hub_iso3": "ESP", "hub_basin": "MEDITERRANEAN", "hub_rank": 2},
        {"hub_port": "Valencia", "hub_iso3": "ESP", "hub_basin": "MEDITERRANEAN", "hub_rank": 3},
        {"hub_port": "Singapore", "hub_iso3": "SGP", "hub_basin": "WESTERN_PACIFIC", "hub_rank": 1},
        {"hub_port": "Jebel Ali", "hub_iso3": "ARE", "hub_basin": "GULF", "hub_rank": 1},
    ]
)

bridge_basin_default_hubs = pd.DataFrame(
    [
        {"origin_basin": "BLACK_SEA", "destination_basin": "ATLANTIC", "hub_basin": "MEDITERRANEAN", "hub_rank": 1},
        {"origin_basin": "BLACK_SEA", "destination_basin": "NORTH_AMERICA_ATLANTIC", "hub_basin": "MEDITERRANEAN", "hub_rank": 1},
        {"origin_basin": "BLACK_SEA", "destination_basin": "WESTERN_PACIFIC", "hub_basin": "MEDITERRANEAN", "hub_rank": 1},
        {"origin_basin": "WEST_AFRICA_ATLANTIC", "destination_basin": "WESTERN_PACIFIC", "hub_basin": "ATLANTIC", "hub_rank": 1},
        {"origin_basin": "MEDITERRANEAN", "destination_basin": "WESTERN_PACIFIC", "hub_basin": "MEDITERRANEAN", "hub_rank": 1},
    ]
)

dim_transshipment_hub.to_parquet(TRANSHIPMENT_HUB_PATH, index=False)
bridge_basin_default_hubs.to_parquet(BASIN_HUB_BRIDGE_PATH, index=False)

display(dim_transshipment_hub)
display(bridge_basin_default_hubs)

,hub_port,hub_iso3,hub_basin,hub_rank
0,Rotterdam,NLD,NORTH_ATLANTIC_EUROPE,1
1,Piraeus,GRC,MEDITERRANEAN,1
2,Algeciras,ESP,MEDITERRANEAN,2
3,Valencia,ESP,MEDITERRANEAN,3
4,Singapore,SGP,WESTERN_PACIFIC,1
5,Jebel Ali,ARE,GULF,1


,origin_basin,destination_basin,hub_basin,hub_rank
0,BLACK_SEA,ATLANTIC,MEDITERRANEAN,1
1,BLACK_SEA,NORTH_AMERICA_ATLANTIC,MEDITERRANEAN,1
2,BLACK_SEA,WESTERN_PACIFIC,MEDITERRANEAN,1
3,WEST_AFRICA_ATLANTIC,WESTERN_PACIFIC,ATLANTIC,1
4,MEDITERRANEAN,WESTERN_PACIFIC,MEDITERRANEAN,1


In [18]:
# extract maritime-eligible route keys for a clean rebuild of the dimension
existing_keys = set()
if (not FULL_REBUILD_DIM_TRADE_ROUTES) and (not dim_trade_routes_existing.empty):
    existing_keys = set(
        dim_trade_routes_existing[["reporter_iso3", "partner_iso3", "partner2_iso3"]]
        .fillna("__NULL__")
        .itertuples(index=False, name=None)
    )

to_route = route_candidates[
    route_candidates["route_applicability_status"] == "MARITIME_ELIGIBLE"
].copy()

to_route["_key"] = list(
    to_route[["reporter_iso3", "partner_iso3", "partner2_iso3"]]
    .fillna("__NULL__")
    .itertuples(index=False, name=None)
)

if existing_keys:
    to_route = to_route[~to_route["_key"].isin(existing_keys)].copy()

display(to_route.head(10))
print("Maritime eligible keys to route:", len(to_route))

,reporter_iso3,partner_iso3,partner2_iso3,row_count,trade_value_usd,has_sea,has_inland_water,has_unknown,has_non_marine,mot_codes_seen,route_applicability_status,_key
0,BGR,AGO,<NA>,16,1.117763e+06,True,False,True,True,0|1000|2100,MARITIME_ELIGIBLE,"(BGR, AGO, __NULL__)"
1,BGR,ALB,<NA>,528,6.187100e+08,True,False,True,True,0|1000|2100|3200,MARITIME_ELIGIBLE,"(BGR, ALB, __NULL__)"
2,BGR,ARE,<NA>,250,1.508292e+08,True,True,True,True,0|1000|2100|2200|3200,MARITIME_ELIGIBLE,"(BGR, ARE, __NULL__)"
3,BGR,ARG,<NA>,341,1.413164e+07,True,False,True,True,0|1000|2100|3100|3200,MARITIME_ELIGIBLE,"(BGR, ARG, __NULL__)"
4,BGR,ARM,<NA>,238,1.966293e+07,True,False,True,True,0|2100|3200,MARITIME_ELIGIBLE,"(BGR, ARM, __NULL__)"
5,BGR,ATG,<NA>,24,5.069904e+05,True,False,True,True,0|2100|3200,MARITIME_ELIGIBLE,"(BGR, ATG, __NULL__)"
6,BGR,AUS,<NA>,106,3.912566e+06,True,False,True,True,0|1000|2100|3200,MARITIME_ELIGIBLE,"(BGR, AUS, __NULL__)"
8,BGR,AZE,<NA>,250,6.370388e+08,True,False,True,True,0|1000|2100|3100|3200,MARITIME_ELIGIBLE,"(BGR, AZE, __NULL__)"
9,BGR,BEL,<NA>,1394,6.942962e+08,True,False,True,True,0|1000|2100|3100|3200|9200,MARITIME_ELIGIBLE,"(BGR, BEL, __NULL__)"
11,BGR,BGD,<NA>,36,5.387316e+08,True,False,True,True,0|1000|2100,MARITIME_ELIGIBLE,"(BGR, BGD, __NULL__)"


Maritime eligible keys to route: 1688


In [19]:
# build dim_trade_routes using motCode-gated, scenario-aware basin graph routing with gateway and hub support
country_ports_clean = (
    dim_country_ports
    .dropna(subset=["iso3", "port_name", "latitude", "longitude"])
    .copy()
)

COUNTRY_PORTS = (
    country_ports_clean.sort_values(["iso3", "port_rank", "port_name"])
    .groupby("iso3")
    .apply(
        lambda g: [
            {
                "port_name": row.port_name,
                "lonlat": (float(row.longitude), float(row.latitude)),
                "port_basin": row.port_basin,
                "port_score": float(row.port_score) if pd.notna(row.port_score) else 0.0,
                "world_water_body": row.world_water_body,
                "fac_container": int(row.fac_container) if pd.notna(row.fac_container) else 0,
                "fac_solid_bulk": int(row.fac_solid_bulk) if pd.notna(row.fac_solid_bulk) else 0,
                "fac_liquid_bulk": int(row.fac_liquid_bulk) if pd.notna(row.fac_liquid_bulk) else 0,
                "fac_oil_terminal": int(row.fac_oil_terminal) if pd.notna(row.fac_oil_terminal) else 0,
                "fac_lng_terminal": int(row.fac_lng_terminal) if pd.notna(row.fac_lng_terminal) else 0,
            }
            for row in g.itertuples(index=False)
        ]
    )
    .to_dict()
)

HUB_BY_BASIN = (
    dim_transshipment_hub.sort_values(["hub_basin", "hub_rank"])
    .groupby("hub_basin")
    .apply(lambda g: g.iloc[0].to_dict())
    .to_dict()
)

GRAPH = {}
for row in basin_graph_edges.itertuples(index=False):
    GRAPH.setdefault(row.origin_basin, []).append(
        {
            "destination_basin": row.destination_basin,
            "chokepoint_name": row.chokepoint_name,
            "scenario_cost": float(row.scenario_cost),
        }
    )

CHOKEPOINT_COORDS = {
    row["chokepoint_name"]: (row["longitude"], row["latitude"])
    for row in dim_chokepoint.dropna(subset=["longitude", "latitude"]).to_dict("records")
}

def _gc_distance_km(a_lonlat, b_lonlat):
    lon1, lat1 = map(math.radians, a_lonlat)
    lon2, lat2 = map(math.radians, b_lonlat)
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    h = math.sin(dlat / 2) ** 2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    return 6371.0088 * (2 * math.asin(math.sqrt(h)))

def _sea_distance_km(a_lonlat, b_lonlat):
    route = sr.searoute(list(a_lonlat), list(b_lonlat), units="km")
    return float(route.properties["length"]), route

def shortest_basin_path(origin_basin: str, destination_basin: str):
    if pd.isna(origin_basin) or pd.isna(destination_basin):
        return [], np.inf
    if origin_basin == destination_basin:
        return [], 0.0

    pq = [(0.0, origin_basin, [], set())]

    while pq:
        cost, basin, cp_path, seen = heapq.heappop(pq)
        if basin == destination_basin:
            return cp_path, cost
        if basin in seen:
            continue
        next_seen = set(seen)
        next_seen.add(basin)

        for edge in GRAPH.get(basin, []):
            next_basin = edge["destination_basin"]
            next_cost = cost + edge["scenario_cost"]
            next_path = cp_path + ([edge["chokepoint_name"]] if edge["chokepoint_name"] != "Open Sea" else [])
            heapq.heappush(pq, (next_cost, next_basin, next_path, next_seen))

    return [], np.inf

def _open_sea_group(origin_basin: str, destination_basin: str) -> str:
    if origin_basin == "BLACK_SEA" or destination_basin == "BLACK_SEA":
        return "BLACK_SEA_REGIONAL"
    if origin_basin in {"MEDITERRANEAN", "NORTH_ATLANTIC_EUROPE", "BALTIC"} and destination_basin in {"MEDITERRANEAN", "NORTH_ATLANTIC_EUROPE", "BALTIC", "ATLANTIC"}:
        return "EUROPEAN_MARITIME"
    return "DIRECT_OR_OPEN_SEA"

def _route_group(cp_sequence: list[str], origin_basin: str, destination_basin: str) -> str:
    if not cp_sequence:
        return _open_sea_group(origin_basin, destination_basin)

    txt = "|".join(cp_sequence).lower()
    if "hormuz" in txt:
        return "HORMUZ_EXPOSED"
    if "suez" in txt:
        return "SUEZ_EXPOSED"
    if "panama" in txt:
        return "PANAMA_EXPOSED"
    if "malacca" in txt:
        return "MALACCA_EXPOSED"
    if "gibraltar" in txt:
        return "GIBRALTAR_EXPOSED"
    if "turkish straits" in txt:
        return "BLACK_SEA_EXIT_EXPOSED"
    if "cape of good hope" in txt:
        return "CAPE_REROUTED"
    return "OTHER_ROUTE"

def choose_gateway_iso3(iso3: str, partner_iso3: str = None):
    gateways = LANDLOCKED_GATEWAYS.get(iso3, [])
    if partner_iso3 in gateways:
        return partner_iso3
    return gateways[0] if gateways else None

def candidate_ports_for_iso3(iso3: str, partner_iso3: str = None):
    flags = COUNTRY_FLAGS.get(iso3, {})
    if flags.get("has_wpi_ports", False):
        return COUNTRY_PORTS.get(iso3, []), None, "DIRECT_PORT"
    if flags.get("is_landlocked_assumed", False):
        gateway_iso3 = choose_gateway_iso3(iso3, partner_iso3)
        if gateway_iso3 and gateway_iso3 in COUNTRY_PORTS:
            return COUNTRY_PORTS[gateway_iso3], gateway_iso3, "SEA_GATEWAY_INFERRED"
        return [], None, "LANDLOCKED_NO_GATEWAY"
    return [], None, "COASTAL_NO_PORT_MATCH"

def choose_best_port_pair(reporter_iso3: str, partner_iso3: str):
    reporter_ports, reporter_gateway_iso3, reporter_basis = candidate_ports_for_iso3(reporter_iso3, partner_iso3)
    partner_ports, partner_gateway_iso3, partner_basis = candidate_ports_for_iso3(partner_iso3, reporter_iso3)

    if not reporter_ports or not partner_ports:
        return None

    best = None
    best_score = -np.inf

    for rp in reporter_ports:
        for pp in partner_ports:
            gc_km = _gc_distance_km(rp["lonlat"], pp["lonlat"])
            cp_sequence, basin_cost = shortest_basin_path(rp["port_basin"], pp["port_basin"])

            # neighbour logic: strongly discourage long chokepoint chains for nearby countries
            neighbour_penalty = 0.0
            if gc_km < 1500 and len(cp_sequence) >= 2:
                neighbour_penalty += 8.0
            if gc_km < 2500 and any(cp in cp_sequence for cp in ["Suez Canal", "Hormuz Strait", "Bab el-Mandeb"]):
                neighbour_penalty += 8.0

            pair_score = 0.0
            pair_score += rp["port_score"] * 0.30
            pair_score += pp["port_score"] * 0.30
            pair_score -= gc_km * 0.001
            pair_score -= basin_cost
            pair_score -= neighbour_penalty

            candidate = {
                "reporter_port": rp["port_name"],
                "partner_port": pp["port_name"],
                "reporter_lonlat": rp["lonlat"],
                "partner_lonlat": pp["lonlat"],
                "reporter_basin": rp["port_basin"],
                "partner_basin": pp["port_basin"],
                "reporter_gateway_iso3": reporter_gateway_iso3,
                "partner_gateway_iso3": partner_gateway_iso3,
                "reporter_port_basis": reporter_basis,
                "partner_port_basis": partner_basis,
                "distance_km": gc_km,
                "cp_sequence": cp_sequence,
                "pair_score": pair_score,
            }

            if pair_score > best_score:
                best = candidate
                best_score = pair_score

    return best

def maybe_assign_hub(origin_basin: str, destination_basin: str):
    hits = bridge_basin_default_hubs[
        (bridge_basin_default_hubs["origin_basin"] == origin_basin)
        & (bridge_basin_default_hubs["destination_basin"] == destination_basin)
    ].sort_values("hub_rank")
    if hits.empty:
        return None
    hub_basin = hits.iloc[0]["hub_basin"]
    return HUB_BY_BASIN.get(hub_basin)

def _forced_path_distance(start_lonlat, end_lonlat, chokepoints):
    if not chokepoints:
        dist_km, route = _sea_distance_km(start_lonlat, end_lonlat)
        return dist_km, route["geometry"]["coordinates"]

    path_nodes = [start_lonlat] + [CHOKEPOINT_COORDS[cp] for cp in chokepoints if cp in CHOKEPOINT_COORDS] + [end_lonlat]
    total_km = 0.0
    stitched = []

    for i in range(len(path_nodes) - 1):
        leg_km, leg_route = _sea_distance_km(path_nodes[i], path_nodes[i + 1])
        total_km += leg_km
        leg_coords = leg_route["geometry"]["coordinates"]
        if i == 0:
            stitched.extend(leg_coords)
        else:
            stitched.extend(leg_coords[1:])

    return total_km, stitched

records = []

for row in to_route.itertuples(index=False):
    r_iso = row.reporter_iso3
    p_iso = row.partner_iso3
    p2_iso = row.partner2_iso3 if pd.notna(row.partner2_iso3) else None

    best = choose_best_port_pair(r_iso, p_iso)
    if best is None:
        records.append(
            {
                "reporter_iso3": r_iso,
                "partner_iso3": p_iso,
                "partner2_iso3": p2_iso,
                "reporter_port": None,
                "partner_port": None,
                "reporter_gateway_iso3": None,
                "partner_gateway_iso3": None,
                "reporter_basin": None,
                "partner_basin": None,
                "distance_km": np.nan,
                "sea_distance_km": np.nan,
                "sea_distance_direct_km": np.nan,
                "sea_distance_forced_km": np.nan,
                "main_chokepoint": None,
                "route_group": "UNROUTED",
                "route_mode": "unrouted",
                "route_basis": "NO_PORT_PATH",
                "route_confidence": "very_low",
                "route_applicability_status": row.route_applicability_status,
                "mot_codes_seen": row.mot_codes_seen,
                "route_scenario": ROUTE_SCENARIO,
                "used_transshipment_hub": False,
                "hub_port": None,
                "hub_iso3": None,
                "hub_basin": None,
                "route_path_coords": [],
            }
        )
        continue

    sea_direct_km = np.nan
    direct_coords = []
    try:
        sea_direct_km, direct_route = _sea_distance_km(best["reporter_lonlat"], best["partner_lonlat"])
        direct_coords = direct_route["geometry"]["coordinates"]
    except Exception:
        pass

    cp_sequence = best["cp_sequence"]
    sea_forced_km = np.nan
    route_coords = direct_coords

    if cp_sequence:
        try:
            sea_forced_km, route_coords = _forced_path_distance(
                best["reporter_lonlat"], best["partner_lonlat"], cp_sequence
            )
        except Exception:
            pass

    sea_km = sea_forced_km if np.isfinite(sea_forced_km) else sea_direct_km
    route_mode = "forced_chokepoint" if cp_sequence else "direct"
    main_cp = cp_sequence[0] if cp_sequence else None

    hub_choice = maybe_assign_hub(best["reporter_basin"], best["partner_basin"])
    used_hub = hub_choice is not None and best["distance_km"] > 2000 and best["reporter_basin"] != best["partner_basin"]

    route_confidence = "medium"
    if best["reporter_port_basis"] != "DIRECT_PORT" or best["partner_port_basis"] != "DIRECT_PORT":
        route_confidence = "low"
    if best["reporter_basin"] == "UNKNOWN_COASTAL" or best["partner_basin"] == "UNKNOWN_COASTAL":
        route_confidence = "very_low"
    if best["distance_km"] < 1200 and any(cp in cp_sequence for cp in ["Suez Canal", "Hormuz Strait", "Bab el-Mandeb", "Panama Canal"]):
        route_confidence = "very_low"

    route_basis_parts = ["SEA_OBSERVED_MOTCODE", ROUTE_SCENARIO]
    if best["reporter_port_basis"] != "DIRECT_PORT":
        route_basis_parts.append(best["reporter_port_basis"])
    if best["partner_port_basis"] != "DIRECT_PORT":
        route_basis_parts.append(best["partner_port_basis"])
    if used_hub:
        route_basis_parts.append("OPTIONAL_HUB_INFERRED")

    records.append(
        {
            "reporter_iso3": r_iso,
            "partner_iso3": p_iso,
            "partner2_iso3": p2_iso,
            "reporter_port": best["reporter_port"],
            "partner_port": best["partner_port"],
            "reporter_gateway_iso3": best["reporter_gateway_iso3"],
            "partner_gateway_iso3": best["partner_gateway_iso3"],
            "reporter_basin": best["reporter_basin"],
            "partner_basin": best["partner_basin"],
            "distance_km": round(best["distance_km"], 2),
            "sea_distance_km": round(sea_km, 2) if np.isfinite(sea_km) else np.nan,
            "sea_distance_direct_km": round(sea_direct_km, 2) if np.isfinite(sea_direct_km) else np.nan,
            "sea_distance_forced_km": round(sea_forced_km, 2) if np.isfinite(sea_forced_km) else np.nan,
            "main_chokepoint": main_cp,
            "route_group": _route_group(cp_sequence, best["reporter_basin"], best["partner_basin"]),
            "route_mode": route_mode,
            "route_basis": "|".join(route_basis_parts),
            "route_confidence": route_confidence,
            "route_applicability_status": row.route_applicability_status,
            "mot_codes_seen": row.mot_codes_seen,
            "route_scenario": ROUTE_SCENARIO,
            "used_transshipment_hub": used_hub,
            "hub_port": hub_choice.get("hub_port") if used_hub else None,
            "hub_iso3": hub_choice.get("hub_iso3") if used_hub else None,
            "hub_basin": hub_choice.get("hub_basin") if used_hub else None,
            "route_path_coords": route_coords,
        }
    )

dim_trade_routes_new = pd.DataFrame.from_records(records)

display(dim_trade_routes_new.head(20))
display(dim_trade_routes_new["route_group"].value_counts(dropna=False).rename_axis("route_group").reset_index(name="route_count"))

,reporter_iso3,partner_iso3,partner2_iso3,reporter_port,partner_port,reporter_gateway_iso3,partner_gateway_iso3,reporter_basin,partner_basin,distance_km,sea_distance_km,sea_distance_direct_km,sea_distance_forced_km,main_chokepoint,route_group,route_mode,route_basis,route_confidence,route_applicability_status,mot_codes_seen,route_scenario,used_transshipment_hub,hub_port,hub_iso3,hub_basin,route_path_coords
0,BGR,AGO,NaN,Varna,Malongo Oil Terminal,None,NaN,BLACK_SEA,SOUTH_ATLANTIC,5637.60,10491.86,10491.86,10491.86,Turkish Straits,GIBRALTAR_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
1,BGR,ALB,NaN,Varna,Durres,None,NaN,BLACK_SEA,MEDITERRANEAN,730.70,1586.65,1586.65,1586.65,Turkish Straits,BLACK_SEA_EXIT_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
2,BGR,ARE,NaN,Varna,Al Fujayrah,None,NaN,BLACK_SEA,INDIAN_OCEAN,3264.57,6930.99,6930.99,6930.99,Turkish Straits,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100|2200|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
3,BGR,ARG,NaN,Varna,Buenos Aires,None,NaN,BLACK_SEA,SOUTH_ATLANTIC,12287.08,13526.48,13526.48,13526.48,Turkish Straits,GIBRALTAR_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100|3100|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
4,BGR,ARM,NaN,Varna,Bandar Abbas,None,IRN,BLACK_SEA,INDIAN_OCEAN,3100.38,7003.54,7003.54,7003.54,Turkish Straits,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest|SEA_GATE...,low,MARITIME_ELIGIBLE,0|2100|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
5,BGR,ATG,NaN,Varna,St Johns,None,NaN,BLACK_SEA,CARIBBEAN,8701.01,9593.88,9593.88,9593.88,Turkish Straits,GIBRALTAR_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|2100|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
6,BGR,AUS,NaN,Varna,Fremantle,None,NaN,BLACK_SEA,INDIAN_OCEAN,12212.65,13871.09,13871.09,13871.09,Turkish Straits,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
7,BGR,AZE,NaN,Varna,Novorossiysk,None,RUS,BLACK_SEA,BLACK_SEA,803.61,849.49,849.49,NaN,NaN,BLACK_SEA_REGIONAL,direct,SEA_OBSERVED_MOTCODE|default_shortest|SEA_GATE...,low,MARITIME_ELIGIBLE,0|1000|2100|3100|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
8,BGR,BEL,NaN,Varna,Antwerpen,None,NaN,BLACK_SEA,NORTH_ATLANTIC_EUROPE,1980.03,6136.35,6136.35,6136.35,Turkish Straits,GIBRALTAR_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100|3100|3200|9200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
9,BGR,BGD,NaN,Varna,Chittagong,None,NaN,BLACK_SEA,INDIAN_OCEAN,6242.61,10533.52,10533.52,10533.52,Turkish Straits,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."


,route_group,route_count
0,DIRECT_OR_OPEN_SEA,449
1,GIBRALTAR_EXPOSED,324
2,SUEZ_EXPOSED,229
3,PANAMA_EXPOSED,181
4,EUROPEAN_MARITIME,132
5,BLACK_SEA_EXIT_EXPOSED,118
6,CAPE_REROUTED,99
7,HORMUZ_EXPOSED,64
8,BLACK_SEA_REGIONAL,63
9,UNROUTED,29


In [21]:
# derive a human-readable basin-path bridge for auditability
audit_pairs = (
    dim_trade_routes_new[["reporter_basin", "partner_basin", "main_chokepoint"]]
    .dropna(subset=["reporter_basin", "partner_basin"])
    .drop_duplicates()
)

audit_records = []
for row in audit_pairs.itertuples(index=False):
    cp_sequence, _ = shortest_basin_path(row.reporter_basin, row.partner_basin)
    if not cp_sequence:
        audit_records.append(
            {
                "origin_basin": row.reporter_basin,
                "destination_basin": row.partner_basin,
                "leg_order": 0,
                "chokepoint_name": "Open Sea / Same Basin",
            }
        )
    else:
        for idx, cp in enumerate(cp_sequence, start=1):
            audit_records.append(
                {
                    "origin_basin": row.reporter_basin,
                    "destination_basin": row.partner_basin,
                    "leg_order": idx,
                    "chokepoint_name": cp,
                }
            )

bridge_port_basin_chokepoints = pd.DataFrame(audit_records).sort_values(
    ["origin_basin", "destination_basin", "leg_order"]
)

bridge_port_basin_chokepoints.to_parquet(BASIN_PATH_RULES_PATH, index=False)
display(bridge_port_basin_chokepoints.head(40))

,origin_basin,destination_basin,leg_order,chokepoint_name
64,ATLANTIC,ARABIAN_SEA,1,Cape of Good Hope
41,ATLANTIC,ATLANTIC,0,Open Sea / Same Basin
57,ATLANTIC,BLACK_SEA,1,Gibraltar Strait
58,ATLANTIC,BLACK_SEA,2,Turkish Straits
38,ATLANTIC,CARIBBEAN,0,Open Sea / Same Basin
59,ATLANTIC,GULF,1,Cape of Good Hope
60,ATLANTIC,GULF,2,Hormuz Strait
54,ATLANTIC,INDIAN_OCEAN,1,Cape of Good Hope
53,ATLANTIC,MEDITERRANEAN,1,Gibraltar Strait
46,ATLANTIC,NORTH_AMERICA_ATLANTIC,0,Open Sea / Same Basin


In [22]:
# merge, deduplicate, and persist the refreshed dim_trade_routes
if FULL_REBUILD_DIM_TRADE_ROUTES or dim_trade_routes_existing.empty:
    dim_trade_routes = dim_trade_routes_new.copy()
else:
    combined = pd.concat(
        [
            dim_trade_routes_existing.reindex(columns=dim_trade_routes_new.columns),
            dim_trade_routes_new,
        ],
        ignore_index=True,
    )

    dim_trade_routes = combined.drop_duplicates(
        subset=["reporter_iso3", "partner_iso3", "partner2_iso3", "route_scenario"],
        keep="last",
    ).sort_values(
        ["reporter_iso3", "partner_iso3", "partner2_iso3", "route_scenario"],
        na_position="last"
    )

dim_trade_routes_export = dim_trade_routes.drop(columns=["route_path_coords"], errors="ignore").copy()
dim_trade_routes_export.to_parquet(DIM_OUTPUT_PATH, index=False)

display(dim_trade_routes["route_mode"].value_counts(dropna=False).rename_axis("route_mode").reset_index(name="route_count"))
display(dim_trade_routes["route_confidence"].value_counts(dropna=False).rename_axis("route_confidence").reset_index(name="route_count"))
display(dim_trade_routes.head(20))
print(DIM_OUTPUT_PATH)

,route_mode,route_count
0,forced_chokepoint,1015
1,direct,644
2,unrouted,29


,route_confidence,route_count
0,medium,1468
1,low,191
2,very_low,29


,reporter_iso3,partner_iso3,partner2_iso3,reporter_port,partner_port,reporter_gateway_iso3,partner_gateway_iso3,reporter_basin,partner_basin,distance_km,sea_distance_km,sea_distance_direct_km,sea_distance_forced_km,main_chokepoint,route_group,route_mode,route_basis,route_confidence,route_applicability_status,mot_codes_seen,route_scenario,used_transshipment_hub,hub_port,hub_iso3,hub_basin,route_path_coords
0,BGR,AGO,NaN,Varna,Malongo Oil Terminal,None,NaN,BLACK_SEA,SOUTH_ATLANTIC,5637.60,10491.86,10491.86,10491.86,Turkish Straits,GIBRALTAR_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
1,BGR,ALB,NaN,Varna,Durres,None,NaN,BLACK_SEA,MEDITERRANEAN,730.70,1586.65,1586.65,1586.65,Turkish Straits,BLACK_SEA_EXIT_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
2,BGR,ARE,NaN,Varna,Al Fujayrah,None,NaN,BLACK_SEA,INDIAN_OCEAN,3264.57,6930.99,6930.99,6930.99,Turkish Straits,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100|2200|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
3,BGR,ARG,NaN,Varna,Buenos Aires,None,NaN,BLACK_SEA,SOUTH_ATLANTIC,12287.08,13526.48,13526.48,13526.48,Turkish Straits,GIBRALTAR_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100|3100|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
4,BGR,ARM,NaN,Varna,Bandar Abbas,None,IRN,BLACK_SEA,INDIAN_OCEAN,3100.38,7003.54,7003.54,7003.54,Turkish Straits,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest|SEA_GATE...,low,MARITIME_ELIGIBLE,0|2100|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
5,BGR,ATG,NaN,Varna,St Johns,None,NaN,BLACK_SEA,CARIBBEAN,8701.01,9593.88,9593.88,9593.88,Turkish Straits,GIBRALTAR_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|2100|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
6,BGR,AUS,NaN,Varna,Fremantle,None,NaN,BLACK_SEA,INDIAN_OCEAN,12212.65,13871.09,13871.09,13871.09,Turkish Straits,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
7,BGR,AZE,NaN,Varna,Novorossiysk,None,RUS,BLACK_SEA,BLACK_SEA,803.61,849.49,849.49,NaN,NaN,BLACK_SEA_REGIONAL,direct,SEA_OBSERVED_MOTCODE|default_shortest|SEA_GATE...,low,MARITIME_ELIGIBLE,0|1000|2100|3100|3200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
8,BGR,BEL,NaN,Varna,Antwerpen,None,NaN,BLACK_SEA,NORTH_ATLANTIC_EUROPE,1980.03,6136.35,6136.35,6136.35,Turkish Straits,GIBRALTAR_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100|3100|3200|9200,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."
9,BGR,BGD,NaN,Varna,Chittagong,None,NaN,BLACK_SEA,INDIAN_OCEAN,6242.61,10533.52,10533.52,10533.52,Turkish Straits,SUEZ_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,MARITIME_ELIGIBLE,0|1000|2100,default_shortest,False,NaN,NaN,None,"[[28.0956, 43.1484], [28.438708, 43.022658], [..."


/Users/chromazone/Documents/Python/Data Enginering Zoomcamp/Capstone_monthly/data/silver/comtrade/dim_trade_routes.parquet


In [24]:
# qa checks for neighbouring-country sanity, gateway behaviour, and scenario consistency
qa_pairs = [
    ("ROU", "BGR"),
    ("ROU", "UKR"),
    ("ROU", "GRC"),
    ("ROU", "ITA"),
    ("BGR", "TUR"),
    ("BGR", "ROU"),
    ("BGR", "GRC"),
    ("BGR", "ITA"),
    ("AUT", "USA"),
    ("CZE", "CHN"),
]

qa_df = (
    dim_trade_routes[
        dim_trade_routes[["reporter_iso3", "partner_iso3"]].apply(tuple, axis=1).isin(qa_pairs)
    ]
    .sort_values(["reporter_iso3", "partner_iso3"])
    .copy()
)

display(
    qa_df[
        [
            "reporter_iso3",
            "partner_iso3",
            "reporter_port",
            "partner_port",
            "reporter_gateway_iso3",
            "partner_gateway_iso3",
            "reporter_basin",
            "partner_basin",
            "main_chokepoint",
            "route_group",
            "route_mode",
            "route_basis",
            "route_confidence",
            "used_transshipment_hub",
            "hub_port",
            "sea_distance_direct_km",
            "sea_distance_forced_km",
            "route_scenario",
        ]
    ]
)

,reporter_iso3,partner_iso3,reporter_port,partner_port,reporter_gateway_iso3,partner_gateway_iso3,reporter_basin,partner_basin,main_chokepoint,route_group,route_mode,route_basis,route_confidence,used_transshipment_hub,hub_port,sea_distance_direct_km,sea_distance_forced_km,route_scenario
40,BGR,GRC,Varna,Piraievs,None,NaN,BLACK_SEA,MEDITERRANEAN,Turkish Straits,BLACK_SEA_EXIT_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,False,NaN,948.26,948.26,default_shortest
51,BGR,ITA,Varna,Brindisi,None,NaN,BLACK_SEA,MEDITERRANEAN,Turkish Straits,BLACK_SEA_EXIT_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,False,NaN,1562.94,1562.94,default_shortest
88,BGR,ROU,Varna,Galati,None,NaN,BLACK_SEA,BLACK_SEA,NaN,BLACK_SEA_REGIONAL,direct,SEA_OBSERVED_MOTCODE|default_shortest,medium,False,NaN,157.38,NaN,default_shortest
104,BGR,TUR,Varna,Samsun,None,NaN,BLACK_SEA,BLACK_SEA,NaN,BLACK_SEA_REGIONAL,direct,SEA_OBSERVED_MOTCODE|default_shortest,medium,False,NaN,819.95,NaN,default_shortest
1359,ROU,BGR,Galati,Varna,None,NaN,BLACK_SEA,BLACK_SEA,NaN,BLACK_SEA_REGIONAL,direct,SEA_OBSERVED_MOTCODE|default_shortest,medium,False,NaN,157.38,NaN,default_shortest
1360,ROU,BGR,Galati,Varna,None,NaN,BLACK_SEA,BLACK_SEA,NaN,BLACK_SEA_REGIONAL,direct,SEA_OBSERVED_MOTCODE|default_shortest,medium,False,NaN,157.38,NaN,default_shortest
1361,ROU,BGR,Galati,Varna,None,NaN,BLACK_SEA,BLACK_SEA,NaN,BLACK_SEA_REGIONAL,direct,SEA_OBSERVED_MOTCODE|default_shortest,medium,False,NaN,157.38,NaN,default_shortest
1362,ROU,BGR,Galati,Varna,None,NaN,BLACK_SEA,BLACK_SEA,NaN,BLACK_SEA_REGIONAL,direct,SEA_OBSERVED_MOTCODE|default_shortest,medium,False,NaN,157.38,NaN,default_shortest
1363,ROU,BGR,Galati,Varna,None,NaN,BLACK_SEA,BLACK_SEA,NaN,BLACK_SEA_REGIONAL,direct,SEA_OBSERVED_MOTCODE|default_shortest,medium,False,NaN,157.38,NaN,default_shortest
1449,ROU,GRC,Galati,Piraievs,None,NaN,BLACK_SEA,MEDITERRANEAN,Turkish Straits,BLACK_SEA_EXIT_EXPOSED,forced_chokepoint,SEA_OBSERVED_MOTCODE|default_shortest,medium,False,NaN,1002.83,1002.83,default_shortest


In [25]:
# visualize a sample of computed routes on an interactive map
sample_size = 30
plot_df = dim_trade_routes[
    dim_trade_routes["route_path_coords"].map(lambda x: isinstance(x, list) and len(x) > 1)
].head(sample_size).copy()

if plot_df.empty:
    print("No route geometries available to plot.")
else:
    route_map = folium.Map(location=[20, 0], zoom_start=2, tiles="CartoDB positron")

    for row in plot_df.itertuples(index=False):
        coords_latlon = [[pt[1], pt[0]] for pt in row.route_path_coords]
        color = "#d62728" if row.route_mode == "forced_chokepoint" else "#1f77b4"
        popup = (
            f"{row.reporter_iso3} ({row.reporter_port}) -> {row.partner_iso3} ({row.partner_port})<br>"
            f"mode={row.route_mode}, cp={row.main_chokepoint}, scenario={row.route_scenario}<br>"
            f"sea_km={row.sea_distance_km}, basis={row.route_basis}"
        )

        folium.PolyLine(coords_latlon, color=color, weight=2, opacity=0.75, popup=popup).add_to(route_map)

    route_map